# DermaVQA — Baseline de Retrieval Textual

Este notebook implementa la primera línea del survey: **retrieval textual antes de fine-tuning**.

La idea es simple y transparente:

1. Indexar las preguntas `question_es` de `train`.
2. Para cada pregunta de `valid`/`test`, recuperar el caso de `train` más parecido.
3. Usar `answer_es` del vecino más cercano como respuesta baseline.
4. Comparar TF-IDF, Multilingual E5 y Sentence-BERT multilingüe.

> Este notebook no entrena modelos y no usa imágenes. Es el baseline barato para decidir si vale la pena avanzar a VLM/LoRA.

## 0. Setup

El notebook puede correr localmente en CPU. Los modelos de embeddings son livianos, pero la primera descarga puede tardar.

Modelos usados:

- TF-IDF con normalización simple.
- `intfloat/multilingual-e5-small`.
- `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`.

In [21]:
# Ejecutá esta celda una vez si estás en un entorno nuevo.
# En Jupyter local, puede pedir reiniciar el kernel después de instalar.
%pip install -q pandas numpy scikit-learn sentence-transformers sacrebleu bert-score tqdm


Note: you may need to restart the kernel to use updated packages.


In [22]:
import json
import math
import os
import re
import shutil
import sys
import unicodedata
import zipfile
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import sacrebleu

def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "outputs" / "datasets").exists() or (
            (candidate / "notebooks").exists() and (candidate / "requirements.txt").exists()
        ):
            return candidate
    return start


PROJECT_ROOT = find_project_root()

print("Python:", sys.version)
print("Working dir:", Path.cwd())
print("Project root:", PROJECT_ROOT)

Python: 3.12.4 | packaged by Anaconda, Inc. | (main, Jun 18 2024, 15:03:56) [MSC v.1929 64 bit (AMD64)]
Working dir: c:\Users\andyd\Udesa\NLP\prueba1\dermavqa-paper\notebooks
Project root: C:\Users\andyd\Udesa\NLP\prueba1\dermavqa-paper


## 1. Cargar dataset enriquecido

Busca primero el JSONL local:

`outputs/datasets/dermavqa_iiyi_llm_synthesized_answer_finetune.jsonl`

Si no existe, intenta encontrar o subir el zip:

`dermavqa_iiyi_llm_synthesized_answer_finetune.zip`

In [23]:
DATASET_JSONL_NAME = "dermavqa_iiyi_llm_synthesized_answer_finetune.jsonl"
DATASET_ZIP_NAME = "dermavqa_iiyi_llm_synthesized_answer_finetune.zip"
EXTRACT_DIR = PROJECT_ROOT / "dermavqa_iiyi_dataset"


def find_upwards(start: Path, relative_path: str) -> Path | None:
    for candidate_root in [start, *start.parents]:
        candidate = candidate_root / relative_path
        if candidate.exists():
            return candidate
    return None


def find_dataset_jsonl() -> Path | None:
    local = PROJECT_ROOT / "outputs" / "datasets" / DATASET_JSONL_NAME
    if local.exists():
        return local
    local = find_upwards(Path.cwd(), f"outputs/datasets/{DATASET_JSONL_NAME}")
    if local:
        return local
    for root in [Path.cwd(), Path("/content"), Path("/kaggle/input")]:
        if root.exists():
            matches = list(root.rglob(DATASET_JSONL_NAME))
            if matches:
                return matches[0]
    return None


def find_dataset_zip() -> Path | None:
    local = PROJECT_ROOT / "outputs" / "datasets" / DATASET_ZIP_NAME
    if local.exists():
        return local
    local = find_upwards(Path.cwd(), f"outputs/datasets/{DATASET_ZIP_NAME}")
    if local:
        return local
    for root in [Path.cwd(), Path("/content"), Path("/kaggle/input")]:
        if root.exists():
            matches = list(root.rglob(DATASET_ZIP_NAME))
            if matches:
                return matches[0]
    return None


jsonl_path = find_dataset_jsonl()

if jsonl_path is None:
    zip_path = find_dataset_zip()
    if zip_path is None:
        try:
            from google.colab import files
            print(f"No encontré {DATASET_JSONL_NAME} ni {DATASET_ZIP_NAME}. Subí el zip ahora.")
            uploaded = files.upload()
            if DATASET_ZIP_NAME not in uploaded:
                raise FileNotFoundError(f"No se subió {DATASET_ZIP_NAME}. Archivos recibidos: {list(uploaded)}")
            zip_path = Path(DATASET_ZIP_NAME)
        except ImportError as error:
            raise FileNotFoundError(
                f"No encontré el dataset. Copiá {DATASET_ZIP_NAME} al directorio actual o agregalo como input."
            ) from error
    print("Extrayendo zip:", zip_path)
    if EXTRACT_DIR.exists():
        shutil.rmtree(EXTRACT_DIR)
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as archive:
        archive.extractall(EXTRACT_DIR)
    matches = list(EXTRACT_DIR.rglob(DATASET_JSONL_NAME))
    if not matches:
        raise FileNotFoundError(f"El zip no contiene {DATASET_JSONL_NAME}")
    jsonl_path = matches[0]

print("Dataset JSONL:", jsonl_path)

Dataset JSONL: C:\Users\andyd\Udesa\NLP\prueba1\dermavqa-paper\outputs\datasets\dermavqa_iiyi_llm_synthesized_answer_finetune.jsonl


In [24]:
with jsonl_path.open("r", encoding="utf-8") as file:
    rows = [json.loads(line) for line in file]

df = pd.DataFrame(rows)
print("Filas por imagen:", len(df))
print("Columnas:", list(df.columns))
display(df.head(3))

Filas por imagen: 2944
Columnas: ['split', 'encounter_id', 'image_id', 'image_path', 'question_es', 'answer_es']


,split,encounter_id,image_id,image_path,question_es,answer_es
0,train,ENC00001,IMG_ENC00001_00001.jpg,C:\Users\andyd\Udesa\NLP\prueba1\dermavqa-pape...,Derrame pleural acompañado de erupción\nUn pac...,"El cuadro es compatible con psoriasis, la cual..."
1,train,ENC00001,IMG_ENC00001_00002.jpg,C:\Users\andyd\Udesa\NLP\prueba1\dermavqa-pape...,Derrame pleural acompañado de erupción\nUn pac...,"El cuadro es compatible con psoriasis, la cual..."
2,train,ENC00002,IMG_ENC00002_00001.jpg,C:\Users\andyd\Udesa\NLP\prueba1\dermavqa-pape...,¿Qué hay en la parte inferior del pie derecho?...,El cuadro es compatible con beriberi.


## 2. Validación y deduplicación por caso

El dataset final está expandido por imagen, pero retrieval textual no usa imágenes. Por eso se evalúa por caso único (`split`, `encounter_id`) para no sobreponderar casos con más fotos.

In [25]:
EXPECTED_ROWS_BY_IMAGE = {"train": 2473, "valid": 157, "test": 314}
EXPECTED_CASES = {"train": 842, "valid": 56, "test": 100}
REQUIRED_COLUMNS = {"split", "encounter_id", "image_id", "image_path", "question_es", "answer_es"}

missing = REQUIRED_COLUMNS - set(df.columns)
assert not missing, f"Faltan columnas: {missing}"

rows_by_image = df["split"].value_counts().sort_index().to_dict()
print("Filas por imagen:", rows_by_image)
assert rows_by_image == dict(sorted(EXPECTED_ROWS_BY_IMAGE.items())), rows_by_image

for column in ["question_es", "answer_es", "image_path"]:
    empty_count = df[column].fillna("").astype(str).str.strip().eq("").sum()
    print(f"{column} vacíos:", empty_count)
    assert empty_count == 0, f"Hay {empty_count} valores vacíos en {column}"

case_df = (
    df.sort_values(["split", "encounter_id", "image_id"])
      .drop_duplicates(["split", "encounter_id"])
      .reset_index(drop=True)
)

cases_by_split = case_df["split"].value_counts().sort_index().to_dict()
print("Casos únicos por split:", cases_by_split)
assert cases_by_split == dict(sorted(EXPECTED_CASES.items())), cases_by_split

train_df = case_df[case_df["split"] == "train"].reset_index(drop=True)
valid_df = case_df[case_df["split"] == "valid"].reset_index(drop=True)
test_df = case_df[case_df["split"] == "test"].reset_index(drop=True)

print("Train cases:", len(train_df))
print("Valid cases:", len(valid_df))
print("Test cases:", len(test_df))

Filas por imagen: {'test': 314, 'train': 2473, 'valid': 157}
question_es vacíos: 0
answer_es vacíos: 0
image_path vacíos: 0
Casos únicos por split: {'test': 100, 'train': 842, 'valid': 56}
Train cases: 842
Valid cases: 56
Test cases: 100


## 3. Textos de índice y configuración

Indexamos solo `question_es` del train. El baseline devuelve la respuesta del caso de train más parecido.

In [26]:
RUN_TFIDF = True
RUN_E5 = True
RUN_SBERT = True

TOP_K = 5
RANDOM_SEED = 42
EMBED_BATCH_SIZE = 64
OUTPUT_DIR = PROJECT_ROOT / "results" / "dataset_enriched" / "retrieval_textual"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

E5_MODEL_NAME = "intfloat/multilingual-e5-small"
SBERT_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"


def clean_text(text: Any) -> str:
    return re.sub(r"\s+", " ", str(text or "")).strip()


train_questions = train_df["question_es"].map(clean_text).tolist()
valid_questions = valid_df["question_es"].map(clean_text).tolist()
test_questions = test_df["question_es"].map(clean_text).tolist()

print("Output dir:", OUTPUT_DIR)
print("Ejemplo train question:", train_questions[0][:500])
print("Ejemplo train answer:", train_df.loc[0, "answer_es"])

Output dir: C:\Users\andyd\Udesa\NLP\prueba1\dermavqa-paper\results\dataset_enriched\retrieval_textual
Ejemplo train question: Derrame pleural acompañado de erupción Un paciente con derrame pleural está acompañado de una erupción sistémica, como se ve en la imagen (actualmente solo está disponible la imagen de la espalda).
Ejemplo train answer: El cuadro es compatible con psoriasis, la cual parece no tener relación con el derrame pleural.


## 4. Helpers de retrieval y métricas

Métricas principales:

- `sacreBLEU`: overlap textual estricto.
- `chrF`: overlap por caracteres, más amable con español y reformulaciones.
- `empty_predictions`: sanity check.
- BERTScore opcional si querés gastar más tiempo de CPU/GPU.

In [27]:
def topk_from_scores(scores: np.ndarray, top_k: int) -> tuple[np.ndarray, np.ndarray]:
    if scores.ndim != 2:
        raise ValueError(f"scores debe ser matriz 2D, recibido {scores.shape}")
    k = min(top_k, scores.shape[1])
    partition = np.argpartition(-scores, kth=np.arange(k), axis=1)[:, :k]
    partition_scores = np.take_along_axis(scores, partition, axis=1)
    order = np.argsort(-partition_scores, axis=1)
    top_indices = np.take_along_axis(partition, order, axis=1)
    top_scores = np.take_along_axis(partition_scores, order, axis=1)
    return top_indices, top_scores


def build_prediction_frame(eval_df: pd.DataFrame, method_name: str, top_indices: np.ndarray, top_scores: np.ndarray) -> pd.DataFrame:
    records = []
    for row_pos, (_, query_row) in enumerate(eval_df.iterrows()):
        best_train_pos = int(top_indices[row_pos, 0])
        best_train_row = train_df.iloc[best_train_pos]
        topk = []
        for rank, (train_pos, score) in enumerate(zip(top_indices[row_pos], top_scores[row_pos]), start=1):
            candidate = train_df.iloc[int(train_pos)]
            topk.append({
                "rank": rank,
                "score": float(score),
                "train_encounter_id": candidate["encounter_id"],
                "train_question_es": clean_text(candidate["question_es"]),
                "train_answer_es": clean_text(candidate["answer_es"]),
            })
        records.append({
            "method": method_name,
            "split": query_row["split"],
            "encounter_id": query_row["encounter_id"],
            "image_id": query_row["image_id"],
            "question_es": clean_text(query_row["question_es"]),
            "reference_answer_es": clean_text(query_row["answer_es"]),
            "retrieved_train_encounter_id": best_train_row["encounter_id"],
            "retrieved_score": float(top_scores[row_pos, 0]),
            "predicted_answer_es": clean_text(best_train_row["answer_es"]),
            "retrieved_question_es": clean_text(best_train_row["question_es"]),
            "topk_json": json.dumps(topk, ensure_ascii=False),
        })
    return pd.DataFrame(records)


def evaluate_predictions(pred_df: pd.DataFrame) -> dict[str, Any]:
    references = pred_df["reference_answer_es"].fillna("").astype(str).tolist()
    hypotheses = pred_df["predicted_answer_es"].fillna("").astype(str).tolist()
    return {
        "rows": len(pred_df),
        "empty_predictions": int(sum(not text.strip() for text in hypotheses)),
        "sacrebleu": float(sacrebleu.corpus_bleu(hypotheses, [references]).score),
        "chrf": float(sacrebleu.corpus_chrf(hypotheses, [references]).score),
        "avg_reference_words": float(np.mean([len(text.split()) for text in references])),
        "avg_prediction_words": float(np.mean([len(text.split()) for text in hypotheses])),
    }


def save_predictions(method_name: str, split_name: str, pred_df: pd.DataFrame) -> dict[str, Any]:
    path = OUTPUT_DIR / f"predictions_{split_name}_{method_name}.csv"
    pred_df.to_csv(path, index=False, encoding="utf-8")
    metrics = evaluate_predictions(pred_df)
    metrics.update({"method": method_name, "split": split_name, "path": str(path.relative_to(PROJECT_ROOT))})
    return metrics

## 5. Baseline TF-IDF (Busca palabras parecidas)

TF-IDF es el baseline transparente: normalización simple, n-gramas 1-2 y similitud coseno.

TF: term frequency, cuántas veces aparece una palabra en un documento.

IDF: inverse document frequency, baja el peso de palabras muy comunes y sube el peso de palabras más distintivas.

Indexamos question_es de train.
Para cada question_es de valid/test, buscamos la pregunta más parecida.
Usamos la answer_es de ese caso recuperado como respuesta baseline

In [28]:
all_metrics = []
predictions: dict[tuple[str, str], pd.DataFrame] = {}

if RUN_TFIDF:
    vectorizer = TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        ngram_range=(1, 2),
        min_df=1,
        max_df=0.95,
        sublinear_tf=True,
        norm="l2",
    )
    train_matrix = vectorizer.fit_transform(train_questions)
    print("TF-IDF train matrix:", train_matrix.shape)

    for split_name, eval_df, eval_questions in [
        ("valid", valid_df, valid_questions),
        ("test", test_df, test_questions),
    ]:
        query_matrix = vectorizer.transform(eval_questions)
        scores = cosine_similarity(query_matrix, train_matrix)
        top_indices, top_scores = topk_from_scores(scores, TOP_K)
        pred_df = build_prediction_frame(eval_df, "tfidf", top_indices, top_scores)
        predictions[("tfidf", split_name)] = pred_df
        all_metrics.append(save_predictions("tfidf", split_name, pred_df))

pd.DataFrame(all_metrics)

TF-IDF train matrix: (842, 28755)


,rows,empty_predictions,sacrebleu,chrf,avg_reference_words,avg_prediction_words,method,split,path
0,56,0,8.675428,30.098917,49.482143,48.303571,tfidf,valid,results\dataset_enriched\retrieval_textual\pre...
1,100,0,8.266186,26.989937,47.830000,39.220000,tfidf,test,results\dataset_enriched\retrieval_textual\pre...


## 6. Retrieval con embeddings densos

- E5 usa prefijos obligatorios: `query: ...` para consultas y `passage: ...` para documentos.
- SBERT MiniLM se usa sin prefijos.
- Ambos normalizan embeddings para que producto punto sea coseno.

In [29]:
def get_device() -> str:
    try:
        import torch
        return "cuda" if torch.cuda.is_available() else "cpu"
    except Exception:
        return "cpu"


def run_sentence_transformer_retrieval(
    method_name: str,
    model_name: str,
    train_texts: list[str],
    eval_items: list[tuple[str, pd.DataFrame, list[str]]],
    train_prefix: str = "",
    query_prefix: str = "",
) -> None:
    from sentence_transformers import SentenceTransformer

    device = get_device()
    print(f"Cargando {model_name} en {device}...")
    model = SentenceTransformer(model_name, device=device)

    corpus_texts = [train_prefix + text for text in train_texts]
    corpus_embeddings = model.encode(
        corpus_texts,
        batch_size=EMBED_BATCH_SIZE,
        normalize_embeddings=True,
        show_progress_bar=True,
        convert_to_numpy=True,
    )
    print(method_name, "corpus embeddings:", corpus_embeddings.shape)

    for split_name, eval_df, eval_texts in eval_items:
        query_texts = [query_prefix + text for text in eval_texts]
        query_embeddings = model.encode(
            query_texts,
            batch_size=EMBED_BATCH_SIZE,
            normalize_embeddings=True,
            show_progress_bar=True,
            convert_to_numpy=True,
        )
        scores = query_embeddings @ corpus_embeddings.T
        top_indices, top_scores = topk_from_scores(scores, TOP_K)
        pred_df = build_prediction_frame(eval_df, method_name, top_indices, top_scores)
        predictions[(method_name, split_name)] = pred_df
        all_metrics.append(save_predictions(method_name, split_name, pred_df))
        print(method_name, split_name, evaluate_predictions(pred_df))


eval_items = [
    ("valid", valid_df, valid_questions),
    ("test", test_df, test_questions),
]

In [30]:
if RUN_E5:
    run_sentence_transformer_retrieval(
        method_name="e5_small",
        model_name=E5_MODEL_NAME,
        train_texts=train_questions,
        eval_items=eval_items,
        train_prefix="passage: ",
        query_prefix="query: ",
    )

Cargando intfloat/multilingual-e5-small en cpu...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/14 [00:00<?, ?it/s]

e5_small corpus embeddings: (842, 384)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

e5_small valid {'rows': 56, 'empty_predictions': 0, 'sacrebleu': 7.939858542053067, 'chrf': 30.534520628391608, 'avg_reference_words': 49.482142857142854, 'avg_prediction_words': 47.035714285714285}


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

e5_small test {'rows': 100, 'empty_predictions': 0, 'sacrebleu': 8.55510311958457, 'chrf': 30.090544824248333, 'avg_reference_words': 47.83, 'avg_prediction_words': 46.43}


In [31]:
if RUN_SBERT:
    run_sentence_transformer_retrieval(
        method_name="sbert_multilingual_minilm",
        model_name=SBERT_MODEL_NAME,
        train_texts=train_questions,
        eval_items=eval_items,
        train_prefix="",
        query_prefix="",
    )

Cargando sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 en cpu...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/14 [00:00<?, ?it/s]

sbert_multilingual_minilm corpus embeddings: (842, 384)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

sbert_multilingual_minilm valid {'rows': 56, 'empty_predictions': 0, 'sacrebleu': 7.917644897970252, 'chrf': 28.005704152974726, 'avg_reference_words': 49.482142857142854, 'avg_prediction_words': 42.410714285714285}


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

sbert_multilingual_minilm test {'rows': 100, 'empty_predictions': 0, 'sacrebleu': 8.794968349242765, 'chrf': 29.81613977617042, 'avg_reference_words': 47.83, 'avg_prediction_words': 44.59}


## 7. Comparación de resultados

La métrica principal para elegir baseline es `valid/chrf` + revisión manual. BLEU puede ser muy duro porque el retrieval devuelve respuestas existentes, no genera una respuesta ajustada al caso.

In [32]:
metrics_df = pd.DataFrame(all_metrics)
metrics_path = OUTPUT_DIR / "metrics_summary.csv"
metrics_df.to_csv(metrics_path, index=False, encoding="utf-8")
print("Métricas guardadas en:", metrics_path)
display(metrics_df.sort_values(["split", "chrf"], ascending=[True, False]))

Métricas guardadas en: C:\Users\andyd\Udesa\NLP\prueba1\dermavqa-paper\results\dataset_enriched\retrieval_textual\metrics_summary.csv


,rows,empty_predictions,sacrebleu,chrf,avg_reference_words,avg_prediction_words,method,split,path
3,100,0,8.555103,30.090545,47.830000,46.430000,e5_small,test,results\dataset_enriched\retrieval_textual\pre...
5,100,0,8.794968,29.816140,47.830000,44.590000,sbert_multilingual_minilm,test,results\dataset_enriched\retrieval_textual\pre...
1,100,0,8.266186,26.989937,47.830000,39.220000,tfidf,test,results\dataset_enriched\retrieval_textual\pre...
2,56,0,7.939859,30.534521,49.482143,47.035714,e5_small,valid,results\dataset_enriched\retrieval_textual\pre...
0,56,0,8.675428,30.098917,49.482143,48.303571,tfidf,valid,results\dataset_enriched\retrieval_textual\pre...
4,56,0,7.917645,28.005704,49.482143,42.410714,sbert_multilingual_minilm,valid,results\dataset_enriched\retrieval_textual\pre...


In [33]:
RUN_BERTSCORE = True

if RUN_BERTSCORE:
    from bert_score import score

    bert_rows = []
    for (method_name, split_name), pred_df in predictions.items():
        print("BERTScore:", method_name, split_name)
        precision, recall, f1 = score(
            pred_df["predicted_answer_es"].tolist(),
            pred_df["reference_answer_es"].tolist(),
            lang="es",
            verbose=True,
            rescale_with_baseline=False,
        )
        bert_rows.append({
            "method": method_name,
            "split": split_name,
            "bertscore_precision": float(precision.mean()),
            "bertscore_recall": float(recall.mean()),
            "bertscore_f1": float(f1.mean()),
        })
    bert_df = pd.DataFrame(bert_rows)
    bert_path = OUTPUT_DIR / "bertscore_summary.csv"
    bert_df.to_csv(bert_path, index=False, encoding="utf-8")
    display(bert_df)
else:
    print("BERTScore desactivado. Cambiá RUN_BERTSCORE=True si querés calcularlo.")

BERTScore: tfidf valid


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


calculating scores...
computing bert embedding.


  0%|          | 0/2 [00:00<?, ?it/s]

...

  0%|          | 0/3 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/2 [00:00<?, ?it/s]

done in 17.11 seconds, 5.84 sentences/sec


,method,split,bertscore_precision,bertscore_recall,bertscore_f1
0,tfidf,valid,0.714604,0.708044,0.709949
1,tfidf,test,0.726601,0.709890,0.716413
2,e5_small,valid,0.711278,0.706671,0.708121
3,e5_small,test,0.716747,0.714838,0.714494
4,sbert_multilingual_minilm,valid,0.728918,0.711750,0.718406
5,sbert_multilingual_minilm,test,0.724194,0.718521,0.719704


## 8. Revisión manual lado a lado

Muestra 10 casos de validación y compara la referencia contra la respuesta recuperada por cada método.

In [34]:
def build_manual_review(split_name: str = "valid", sample_size: int = 10, seed: int = RANDOM_SEED) -> pd.DataFrame:
    base = valid_df if split_name == "valid" else test_df
    sample_ids = (
        base.sample(min(sample_size, len(base)), random_state=seed)["encounter_id"]
        .tolist()
    )
    rows = []
    methods = sorted({method for method, split in predictions if split == split_name})
    for encounter_id in sample_ids:
        query_row = base[base["encounter_id"] == encounter_id].iloc[0]
        record = {
            "split": split_name,
            "encounter_id": encounter_id,
            "question_es": clean_text(query_row["question_es"]),
            "reference_answer_es": clean_text(query_row["answer_es"]),
        }
        for method in methods:
            pred_df = predictions[(method, split_name)]
            pred_row = pred_df[pred_df["encounter_id"] == encounter_id].iloc[0]
            record[f"{method}_retrieved_id"] = pred_row["retrieved_train_encounter_id"]
            record[f"{method}_score"] = pred_row["retrieved_score"]
            record[f"{method}_answer_es"] = pred_row["predicted_answer_es"]
        rows.append(record)
    return pd.DataFrame(rows)

manual_valid_df = build_manual_review("valid", 10)
manual_path = OUTPUT_DIR / "manual_review_valid_10.csv"
manual_valid_df.to_csv(manual_path, index=False, encoding="utf-8")
print("Revisión manual guardada en:", manual_path)
display(manual_valid_df)

Revisión manual guardada en: C:\Users\andyd\Udesa\NLP\prueba1\dermavqa-paper\results\dataset_enriched\retrieval_textual\manual_review_valid_10.csv


,split,encounter_id,question_es,reference_answer_es,e5_small_retrieved_id,e5_small_score,e5_small_answer_es,sbert_multilingual_minilm_retrieved_id,sbert_multilingual_minilm_score,sbert_multilingual_minilm_answer_es,tfidf_retrieved_id,tfidf_score,tfidf_answer_es
0,valid,ENC00852,¿Es vitíligo? Ver la foto. La paciente es una ...,El cuadro es compatible con leucoplasia secund...,ENC00711,0.906236,"El cuadro es compatible con psoriasis, pitiria...",ENC00711,0.810208,"El cuadro es compatible con psoriasis, pitiria...",ENC00108,0.142736,El cuadro es compatible con dermatitis de cont...
1,valid,ENC00857,¿Podrían los expertos echar un vistazo? Mujer ...,El cuadro es compatible con hipopigmentación p...,ENC00203,0.909304,El cuadro es compatible con síndrome de incont...,ENC00444,0.751897,El cuadro es compatible con pie de atleta. Se ...,ENC00535,0.285490,"El cuadro es compatible con neurofibromatosis,..."
2,valid,ENC00885,Ver foto para diagnóstico. Paciente varón de 1...,El cuadro es compatible con impétigo o con ecz...,ENC00499,0.920313,El cuadro es compatible con atrofia primaria m...,ENC00024,0.788046,El cuadro es compatible con urticaria colinérg...,ENC00694,0.129449,El cuadro es compatible con varias entidades c...
3,valid,ENC00865,"Por favor, indíqueme de qué tipo de enfermedad...","El cuadro es compatible con psoriasis, una enf...",ENC00107,0.912903,El cuadro es compatible con neurodermatitis. E...,ENC00759,0.722581,El cuadro es compatible con dermatitis por est...,ENC00207,0.359331,"El cuadro es compatible con eczema, una enferm..."
4,valid,ENC00871,enfermedades hereditarias de la piel: ¿podéis ...,"El cuadro es compatible con poroqueratosis, di...",ENC00217,0.894723,El cuadro es compatible con atrofia cutánea se...,ENC00632,0.790063,"El cuadro es compatible con neurodermatitis, q...",ENC00554,0.187054,"El cuadro es compatible con urticaria, probabl..."
5,valid,ENC00902,Ayuden a diagnosticar juntos la enfermedad. Ap...,"El cuadro es compatible con eczema, psoriasis ...",ENC00172,0.899499,"El cuadro es compatible con psoriasis, que se ...",ENC00498,0.651018,"El cuadro es compatible con eczema, posiblemen...",ENC00809,0.160744,El cuadro es compatible con eritema multiforme...
6,valid,ENC00888,¿Sabéis que es el bulto rojo de la espalda? ¿Q...,El cuadro es compatible con dermatitis estival...,ENC00617,0.883527,El cuadro es compatible con urticaria papular;...,ENC00760,0.633127,El cuadro es compatible con eczema o dermatiti...,ENC00658,0.150799,El cuadro no parece psoriasis; entre los diagn...
7,valid,ENC00878,"Paciente de hace 3 meses. Mujer, 60+ años. Más...","El cuadro es compatible con eczema agrietado, ...",ENC00279,0.903686,El cuadro es compatible con una lesión cutánea...,ENC00730,0.763815,El cuadro es compatible con dermatitis solar o...,ENC00590,0.147527,El cuadro es compatible con nódulos reumatoides.
8,valid,ENC00896,Parece que las manos de la chica hayan estado ...,El cuadro es compatible con hiperhidrosis palm...,ENC00056,0.880324,El cuadro es compatible con una erupción cután...,ENC00451,0.627389,El cuadro es compatible con eczema y presenta ...,ENC00070,0.235736,El cuadro es compatible con un nevus pigmentad...
9,valid,ENC00864,¿Qué es ésto? ¿Podéis mirarlo pronto? Urgente....,"El cuadro es compatible con urticaria, probabl...",ENC00501,0.891251,El cuadro es compatible con esporotricosis; si...,ENC00049,0.620209,El cuadro es compatible con una dermatitis por...,ENC00222,0.156105,"El cuadro es compatible con urticaria aguda, i..."


In [35]:
for _, row in manual_valid_df.iterrows():
    print("\n" + "=" * 120)
    print("encounter_id:", row["encounter_id"])
    print("\nPregunta:")
    print(row["question_es"][:900])
    print("\nReferencia:")
    print(row["reference_answer_es"])
    for method in sorted({method for method, split in predictions if split == "valid"}):
        print(f"\n[{method}] retrieved_id={row[f'{method}_retrieved_id']} score={row[f'{method}_score']:.4f}")
        print(row[f"{method}_answer_es"])


encounter_id: ENC00852

Pregunta:
¿Es vitíligo? Ver la foto. La paciente es una mujer de mediana edad, de unos 50 años. Presenta erupciones de color rojo oscuro en el dorso de la mano. Se convierten gradualmente en leucoplasia, pero se tornan rojizas al frotar. La paciente no siente síntomas específicos, pero presenta problemas mentales. Tratamiento basado en el diagnóstico como eczema y vitíligo. Sin embargo, el síntoma empeora: las erupciones aumentan de tamaño y un mes después aparecen máculas de color rojo claro en la cara, sin escamas cutáneas. La paciente se niega a someterse a un examen patológico.

Referencia:
El cuadro es compatible con leucoplasia secundaria, que puede ser temporal y resolverse en 6 a 12 meses; sin embargo, la presencia de pérdida de pigmentación tras inflamación también sugiere la posibilidad de vitíligo, aunque en el vitíligo típico las manchas son blanquecinas desde el inicio y con límites claros, y una respuesta indica que no es vitíligo en este caso.

[

## 9. Inspeccionar top-k de un caso

Sirve para entender si el error viene de falta de casos similares o de mala representación textual.

In [36]:
METHOD_TO_INSPECT = "e5_small" if ("e5_small", "valid") in predictions else sorted(predictions.keys())[0][0]
SPLIT_TO_INSPECT = "valid"
ROW_TO_INSPECT = 0

pred_df = predictions[(METHOD_TO_INSPECT, SPLIT_TO_INSPECT)]
row = pred_df.iloc[ROW_TO_INSPECT]
print("Método:", METHOD_TO_INSPECT)
print("Caso:", row["encounter_id"])
print("\nPregunta query:")
print(row["question_es"])
print("\nReferencia:")
print(row["reference_answer_es"])
print("\nTop-k recuperado:")
for item in json.loads(row["topk_json"]):
    print("\n--- rank", item["rank"], "score", round(item["score"], 4), "train", item["train_encounter_id"])
    print("Q:", item["train_question_es"][:700])
    print("A:", item["train_answer_es"])

Método: e5_small
Caso: ENC00852

Pregunta query:
¿Es vitíligo? Ver la foto. La paciente es una mujer de mediana edad, de unos 50 años. Presenta erupciones de color rojo oscuro en el dorso de la mano. Se convierten gradualmente en leucoplasia, pero se tornan rojizas al frotar. La paciente no siente síntomas específicos, pero presenta problemas mentales. Tratamiento basado en el diagnóstico como eczema y vitíligo. Sin embargo, el síntoma empeora: las erupciones aumentan de tamaño y un mes después aparecen máculas de color rojo claro en la cara, sin escamas cutáneas. La paciente se niega a someterse a un examen patológico.

Referencia:
El cuadro es compatible con leucoplasia secundaria, que puede ser temporal y resolverse en 6 a 12 meses; sin embargo, la presencia de pérdida de pigmentación tras inflamación también sugiere la posibilidad de vitíligo, aunque en el vitíligo típico las manchas son blanquecinas desde el inicio y con límites claros, y una respuesta indica que no es vitíligo en

## 10. Exportar resultados

Comprime métricas y predicciones para conservar el baseline.

In [37]:
archive_path = OUTPUT_DIR / "retrieval_textual_results.zip"
if archive_path.exists():
    archive_path.unlink()

with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUTPUT_DIR.glob("*")):
        if path.resolve() == archive_path.resolve() or not path.is_file():
            continue
        archive.write(path, arcname=path.name)

print("Resultados comprimidos:", archive_path)
print("Archivos generados:")
for path in sorted(OUTPUT_DIR.glob("*")):
    print("-", path, path.stat().st_size)

try:
    from google.colab import files
    files.download(str(archive_path))
except Exception:
    print("Si estás local/Kaggle, descargá o guardá:", archive_path)


Resultados comprimidos: C:\Users\andyd\Udesa\NLP\prueba1\dermavqa-paper\results\dataset_enriched\retrieval_textual\retrieval_textual_results.zip
Archivos generados:
- C:\Users\andyd\Udesa\NLP\prueba1\dermavqa-paper\results\dataset_enriched\retrieval_textual\bertscore_summary.csv 524
- C:\Users\andyd\Udesa\NLP\prueba1\dermavqa-paper\results\dataset_enriched\retrieval_textual\manual_review_valid_10.csv 19040
- C:\Users\andyd\Udesa\NLP\prueba1\dermavqa-paper\results\dataset_enriched\retrieval_textual\metrics_summary.csv 1090
- C:\Users\andyd\Udesa\NLP\prueba1\dermavqa-paper\results\dataset_enriched\retrieval_textual\predictions_test_e5_small.csv 670072
- C:\Users\andyd\Udesa\NLP\prueba1\dermavqa-paper\results\dataset_enriched\retrieval_textual\predictions_test_sbert_multilingual_minilm.csv 637819
- C:\Users\andyd\Udesa\NLP\prueba1\dermavqa-paper\results\dataset_enriched\retrieval_textual\predictions_test_tfidf.csv 561045
- C:\Users\andyd\Udesa\NLP\prueba1\dermavqa-paper\results\dataset_en

## 11. Interpretación esperada

- Si TF-IDF gana o empata, el dataset tiene mucha señal léxica y conviene mantenerlo como baseline fuerte.
- Si E5/SBERT ganan, hay señal semántica útil y conviene considerar retrieval semántico como componente del sistema.
- Si todos fallan en muchos casos, probablemente el paso siguiente sea VLM: la imagen aporta información que el texto solo no captura.